# UCInsure: Flood Model

An end-to-end actuarial ML pipeline combining flood-risk data ingestion, exploratory analysis, machine learning, and visualization using **FEMA NFIP** claims data.

## Pipeline Stages

| # | Stage | Description |
|---|-------|-------------|
| 1 | **Setup & Imports** | Install dependencies and configure global constants |

## Data Source
- FEMA OpenFEMA API v2 — `FimaNfipClaims`
- Cached at `data/cache/FimaNfipClaimsV2.csv` after first run
- Sample size: 250,000 rows (configurable via `SAMPLE_ROWS`)

## 1. Setup & Imports

Install required packages and configure global constants shared across all pipeline stages.

In [ ]:
%pip install --break-system-packages numpy pandas scikit-learn joblib matplotlib seaborn

import json
import math
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import joblib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ── Global constants ──────────────────────────────────────────────────────────
DATA_URL        = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
DATA_CACHE_PATH = Path("data") / "cache" / "FimaNfipClaimsV2.csv"
MODEL_CACHE_DIR = Path("data") / "cache" / "models"

ALLOWED_COLUMNS = (
    "reportedCity",
    "reportedZipCode",
    "latitude",
    "longitude",
    "floodEvent",
    "dateOfLoss",
    "yearOfLoss",
    "floodZoneCurrent",
    "waterDepth",
    "numberOfFloorsInTheInsuredBuilding",
    "occupancyType",
    "primaryResidenceIndicator",
    "buildingPropertyValue",
    "contentsPropertyValue",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "buildingDamageAmount",
)

SAMPLE_ROWS  = 250_000
RANDOM_STATE = 42

print("✓ Imports and constants ready.")
